# 📊 Exploration des données (EDA) — Obesity Classification

Ce notebook est dédié uniquement à l'exploration et la visualisation des données.  
L'entraînement des modèles se fait via le script `main.py`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from src.data.load import load_data
from src.data.preprocess import add_basic_features

from src.utils import ORDINAL_MAPPING

## 1. Chargement des données

In [ ]:
df = load_data('data/ObesityDataSet_raw_and_data_sinthetic.csv')
df.head()

In [ ]:
print(f"Shape: {df.shape}")
print("\nDistribution de la target (NObeyesdad):")
print(df.NObeyesdad.value_counts())
print(f"\nValeurs manquantes:\n{df.isna().sum()[df.isna().sum() > 0]}")
if df.isna().sum().sum() == 0:
    print("Aucune valeur manquante.")
print(f"\nTypes:\n{df.dtypes}")

## 2. Feature engineering exploratoire

In [ ]:
df = add_basic_features(df)
df['NObeyesdad_ord'] = df['NObeyesdad'].map(ORDINAL_MAPPING)

## 3. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col, title in zip(
    axes,
    ['BMI', 'Weight', 'Height'],
    ['Distribution du BMI', 'Distribution du Weight', 'Distribution du Height'],
):
    sns.boxplot(x='NObeyesdad', y=col, data=df, order=ORDINAL_MAPPING.keys(), ax=ax)
    ax.set_title(f'{title} selon NObeyesdad')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Corrélation de Spearman
corr, pval = spearmanr(df['BMI'], df['NObeyesdad_ord'])
print(f"Corrélation de Spearman BMI vs NObeyesdad_ord : {corr:.3f} (p={pval:.2e})")

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x='Height', y='Weight', hue='NObeyesdad', data=df, palette='tab10', alpha=0.7)
plt.title('Poids vs Taille selon la classe NObeyesdad')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Conclusion EDA

- **Weight** et **Height** (et donc **BMI**) sont directement corrélés à la target → supprimés pour l’entraînement.
- La classification est **ordinale** : les 7 classes suivent un ordre naturel.
- Aucune valeur manquante.
- Certaines colonnes ordinales (FAF, FCVC, NCP, TUE, CH2O) ont des valeurs non entières (artefact SMOTE) → arrondies.